In [ ]:
from collections import defaultdict
import re
import os
from dataclasses import dataclass, field
from typing import Dict, List, Optional

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from scipy.stats import kendalltau

# ── Style ─────────────────────────────────────────────────────────────────────

STYLE = {
    'font.family':      'serif',
    'font.size':        11,
    'axes.linewidth':   1.2,
    'xtick.direction':  'in',
    'ytick.direction':  'in',
    'grid.alpha':       0.3,
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'text.color':       'black',
    'axes.labelcolor':  'black',
    'xtick.color':      'black',
    'ytick.color':      'black',
    'axes.edgecolor':   'black',
}

BLUE = "#1a6fba"
RED  = "#c0392b"


# ── Data class ────────────────────────────────────────────────────────────────

@dataclass
class Block:
    layers:  List[int]
    cosine:  Dict[int, float] = field(default_factory=dict)
    pos_acc: Optional[float]  = None
    neg_acc: Optional[float]  = None

    @property
    def anchor(self) -> int:
        """Smallest layer index in this block (identifies the block uniquely)."""
        return min(self.layers)


class Reader:
    """Parse a power-seeking probe log file into structured data."""

    _block_re = re.compile(r"\[Layers:\] (\[[\d, ]+\])")
    _layer_re = re.compile(r"\[Layer:\] (\d+) \| Cosine Distance \[Mean:\] ([\d.]+)")
    _acc_re   = re.compile(r"\[Positive Accuracy:\] ([\d.]+) \[Negative Accuracy:\] ([\d.]+)")

    def __init__(self, log_path: str):
        self.log_path = log_path
        self.blocks: List[Block] = []
        self.n_layers: int = 0
        self._parse()
        self._build_matrices()

    # ── Parsing ───────────────────────────────────────────────────────────────

    def _parse(self):
        blocks, current = [], None
        with open(self.log_path) as f:
            for line in f:
                line = line.strip()
                m = self._block_re.match(line)
                if m:
                    if current is not None:
                        blocks.append(current)
                    layer_list = list(map(int, re.findall(r"\d+", m.group(1))))
                    current = Block(layers=layer_list)
                    continue
                if current is None:
                    continue
                m = self._layer_re.match(line)
                if m:
                    current.cosine[int(m.group(1))] = float(m.group(2))
                    continue
                m = self._acc_re.match(line)
                if m:
                    current.pos_acc = float(m.group(1))
                    current.neg_acc = float(m.group(2))
        if current is not None:
            blocks.append(current)
        self.blocks = blocks

    # ── Matrix building ───────────────────────────────────────────────────────

    def _build_matrices(self):
        N = max(max(b.layers) for b in self.blocks) + 1
        self.n_layers = N

        heat   = np.full((N, N), np.nan)
        active = np.zeros((N, N), dtype=bool)

        pos_by_anchor: Dict[int, float] = {}
        neg_by_anchor: Dict[int, float] = {}

        self.anchor_cost = np.full(N, np.nan)
        self.anchor_gain = np.full(N, np.nan)
        for b in self.blocks:
            a = b.anchor
            pos_by_anchor[a] = b.pos_acc
            neg_by_anchor[a] = b.neg_acc
            for idx in b.layers:
                active[idx, a] = True
            for idx, val in b.cosine.items():
                heat[idx, a] = val

            sorted_layers = sorted(b.cosine.keys())
            distances = [b.cosine[idx] for idx in sorted_layers if b.cosine[idx] != 1.0]
            self.anchor_cost[N-a-1] = distances[-1]

        self.heat   = heat    # (N, N)  row=measured layer, col=anchor
        self.active = active  # (N, N)

        anchors = sorted(pos_by_anchor)
        self.anchors    = anchors
        self.pos_acc    = [pos_by_anchor[a] for a in anchors]
        self.neg_acc    = [neg_by_anchor[a] for a in anchors]
        
        for a in range(N):
            self.anchor_gain[a] = (self.pos_acc[N-a-1] + self.neg_acc[N-a-1]) /2
        # Optional: Print the results immediately
        # print("---Layer Score---")
        # for a in self.anchors: # 32 -> 1 descending
        #     sens = self.anchor_cost[a]
        #     gain = self.anchor_gain[a]
        #     if not np.isnan(sens):
        #         print(f"Steered Layer {N-a:02d}: Cost = {sens:+.4f} Gain = {gain:+4f}")

# ── Plotting ──────────────────────────────────────────────────────────────────
from scipy.ndimage import gaussian_filter1d
from scipy import stats
def _build_heatmap_rgba(
    heat: np.ndarray,
    active: np.ndarray,
    cmap: LinearSegmentedColormap,
) -> tuple:
    """Return (rgba, norm) with inactive cells painted white."""
    N= len(heat)
    nan_mask = np.tril(np.ones((N, N)))
    nan_mask[nan_mask == 0] = np.nan 
    
    masked_heat = heat * nan_mask
    transformed_mask = masked_heat.T[:, ::-1]
    distance_cost = pareto(np.nanmean(transformed_mask, axis=1)[::-1])
    layer_weight = np.nanmean(transformed_mask, axis=0).tolist()
    
    layer_weight.reverse()
    normalized = 1-min_max_normalize(layer_weight)
    final = np.where(distance_cost == 0, normalized ** 1, normalized)
    print("Distance Cost (First Col):", distance_cost.tolist())
    print("Layer Weights (Row Averages):",normalized.tolist())
    print("Final Weights :",final.tolist())
    
    
    vmin = float(np.nanmin(heat[active]))
    vmax = 1.0
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    rgba = cmap(norm(heat))
    rgba[~active] = [1.0, 1.0, 1.0, 1.0]
    return rgba, norm

def pareto(data, fill_value=0.0):
    cleaned = data.copy()
    
    if cleaned.ndim == 1:
        current_min = cleaned[0]
        for i in range(1, len(cleaned)):
            if cleaned[i] >= current_min:
                cleaned[i] = fill_value
            else:
                current_min = cleaned[i]
                cleaned[i] = 1.
    cleaned[0] = 1.
    return cleaned

def _draw_heatmap(ax, fig, rgba: np.ndarray, norm, cmap, N: int):
    """
    Display the heatmap with:
      x = Steered Layer (32 on left → 1 on right)
      y = Layer Index   (0 at bottom → 31 at top, displayed 31 top → 0 bottom)
    Mask (white) is top-right; data is bottom-left.
    """
    # Transpose so (x=steered, y=layer_index), then flip both axes for orientation
    display = rgba.transpose(1, 0, 2)[:, ::-1, :]
    ax.imshow(display, aspect="auto", origin="lower", interpolation="nearest")

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.04)
    cbar.set_label("Cosine Distance (Mean)", fontsize=10)
    cbar.ax.yaxis.set_tick_params(color="black", labelsize=8)
    cbar.outline.set_edgecolor("black")
    cbar.outline.set_linewidth(1.0)

    ticks = list(range(N))

    # X ticks: steered layer 32→1 (left to right)
    x_labels = [i for i in range(N)]
    ax.set_xticks(ticks)
    ax.set_xticklabels(x_labels, fontsize=7)

    # Y ticks: layer index 31→0 (top to bottom, origin=lower so reversed)
    y_labels = [str(N - 1 - i) for i in range(N)]
    ax.set_yticks(ticks)
    ax.set_yticklabels(y_labels, fontsize=7)

    ax.tick_params(axis="both", which="both", direction="in", length=3)
    ax.set_xlabel("Layer Index  (0 → 31)", fontsize=11, labelpad=6)
    ax.set_ylabel("Layer Index  (31 → 0)", fontsize=11, labelpad=6)
    ax.set_title("Cosine-Distance Heatmap", fontsize=13, pad=8, fontweight="bold")

    # Faint grid every 4
    for v in range(0, N, 4):
        ax.axvline(v - 0.5, color="black", lw=0.4, alpha=0.15)
        ax.axhline(v - 0.5, color="black", lw=0.4, alpha=0.15)

    ax.text(0.99, 0.99, "white = inactive",
            transform=ax.transAxes, fontsize=7.5, color="#555",
            ha="right", va="top")


def _draw_accuracy(ax, anchors: List[int], pos_acc: List[float], neg_acc: List[float]):
    """Line chart: x = anchor layer (31→0), y = accuracy."""
    anchors_rev = anchors[::-1]
    pos_rev     = pos_acc[::-1]
    neg_rev     = neg_acc[::-1]

    ax.plot(anchors_rev, pos_rev, color=BLUE, lw=2.0, marker="o",
            markersize=4.5, label="Positive Accuracy", zorder=3)
    ax.plot(anchors_rev, neg_rev, color=RED,  lw=2.0, marker="s",
            markersize=4.5, label="Negative Accuracy", zorder=3)
    ax.fill_between(anchors_rev, pos_rev, neg_rev, alpha=0.10, color="grey")

    ax.set_xlim(max(anchors), min(anchors))   # 31 on left, 0 on right
    ax.set_xticks(list(range(max(anchors) + 1)))
    ax.set_xticklabels([str(i) for i in range(max(anchors)+1, 0, -1)])
    ax.set_xlabel("Steered Layer  (1 → 32)", fontsize=10, labelpad=6)
    ax.set_ylabel("Accuracy", fontsize=10, labelpad=6)
    ax.set_title("Probe Accuracy vs. Layer Index", fontsize=12, pad=8, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.tick_params(axis="both", which="both", direction="in", length=4)
    ax.grid(True, color="grey", lw=0.6, linestyle="--", alpha=0.3)
    ax.legend(framealpha=0.9, facecolor="white", edgecolor="black",
              fontsize=9, loc="upper left")

    # Annotate peaks
    best_i = int(np.argmax(pos_rev))
    ax.annotate(f"{pos_rev[best_i]:.1%}",
        xy=(anchors_rev[best_i], pos_rev[best_i]),
        xytext=(anchors_rev[best_i] + 5, pos_rev[best_i] - 0.12),
        color=BLUE, fontsize=8.5,
        arrowprops=dict(arrowstyle="->", color=BLUE, lw=0.9))

    best_j = int(np.argmax(neg_rev))
    ax.annotate(f"{neg_rev[best_j]:.1%}",
        xy=(anchors_rev[best_j], neg_rev[best_j]),
        xytext=(anchors_rev[best_j] + 5, neg_rev[best_j] + 0.05),
        color=RED, fontsize=8.5,
        arrowprops=dict(arrowstyle="->", color=RED, lw=0.9))


def plot_analysis(
    dataset: Reader,
    output_path: str,
    title: Optional[str] = None,
    figsize: tuple = (24, 10),
    dpi: int = 160,
):
    """
    Render the heatmap + accuracy line chart and save to output_path.

    Parameters
    ----------
    dataset     : PowerSeekingData  parsed log data
    output_path : str               path for the saved PNG
    title       : str, optional     figure suptitle (auto-generated from log filename if None)
    figsize     : tuple             matplotlib figure size
    dpi         : int               output resolution
    """
    plt.rcParams.update(STYLE)

    # Colormap: truncated plasma (dark purple → magenta → orange, no yellow)
    plasma_colors = plt.cm.plasma(np.linspace(0.0, 0.80, 256))
    cmap = LinearSegmentedColormap.from_list("plasma_trunc", plasma_colors)

    rgba, norm = _build_heatmap_rgba(dataset.heat, dataset.active, cmap)

    # Auto title from filename
    if title is None:
        base = os.path.splitext(os.path.basename(dataset.log_path))[0]
        title = base.replace("_", " ")

    fig = plt.figure(figsize=figsize, facecolor="white")
    fig.suptitle(title, fontsize=16, fontweight="bold", y=0.99)

    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.6, 1],
                  wspace=0.18, left=0.07, right=0.97, top=0.93, bottom=0.11)
    ax_heat = fig.add_subplot(gs[0])
    ax_line = fig.add_subplot(gs[1])

    _draw_heatmap(ax_heat, fig, rgba, norm, cmap, dataset.n_layers)
    _draw_accuracy(ax_line, dataset.anchors, dataset.pos_acc, dataset.neg_acc)

    os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else ".", exist_ok=True)
    fig.savefig(output_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    print(f"Saved → {output_path}")
    plt.close(fig)


def min_max_normalize(data, round_decimals=4):
    """
    Scales a list/array so the minimum value is 0.0 and the maximum is 1.0.
    """
    arr = np.array(data)
    val_min = np.min(arr)
    val_max = np.max(arr)
    
    # Safeguard against division by zero if all values are identical
    if val_max == val_min:
        return np.zeros_like(arr).tolist()
        
    normalized = (arr - val_min) / (val_max - val_min)
    
    if round_decimals is not None:
        normalized = np.round(normalized, round_decimals)
        
    return normalized

def abs_z_score_normalize(data, round_decimals=4):
    """
    Calculates the absolute Z-score for a list/array.
    Formula: |(x - mean) / standard_deviation|
    """
    arr = np.array(data)
    val_mean = np.mean(arr)
    val_std = np.std(arr)
    
    # Safeguard against division by zero if all values are identical
    if val_std == 0:
        # Return an array of zeros (or list) if there is no variance
        return np.zeros_like(arr, dtype=float).tolist() if isinstance(data, list) else np.zeros_like(arr, dtype=float)
        
    # Calculate absolute Z-score
    abs_z_scores = (arr - val_mean) / val_std
    
    if round_decimals is not None:
        abs_z_scores = np.round(abs_z_scores, round_decimals)
        
        
    return abs_z_scores



In [ ]:
LOG_FILE = [
    "/home/yosef/ws/thesis/scrip/log/Mistral-7B-Instruct-v0.3_power-seeking_log.txt",
    "/home/yosef/ws/thesis/scrip/log/Mistral-7B-Instruct-v0.3_wealth-seeking_log.txt",
    "/home/yosef/ws/thesis/scrip/log/Llama-3.1-8B-Instruct_wealth-seeking_log.txt",
    "/home/yosef/ws/thesis/scrip/log/Llama-3.1-8B-Instruct_power-seeking_log.txt"


]
OUTPUT   = [
    "outputs/mistral_power_seeking_refactored.png",
    "outputs/mistral_wealth_refactored.png",
    "outputs/llama_wealth_refactored.png",
    "outputs/llama_power_seeking_refactored.png",
            
]

for file,img in zip(LOG_FILE,OUTPUT):
    dataset = Reader(file)
    # Generate Plots
    plot_analysis(dataset, img)
    print()

Distance Cost (First Col): [1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Layer Weights (Row Averages): [0.0, 0.21299999999999997, 0.38970000000000005, 0.372, 0.3893, 0.43069999999999997, 0.4505, 0.5399, 0.6135999999999999, 0.6303000000000001, 0.7565999999999999, 0.8293, 0.7028, 0.5339, 0.6772, 0.7459, 0.7142999999999999, 0.47019999999999995, 0.7025, 0.6126, 0.6778, 0.7113, 0.8852, 0.8366, 1.0, 0.9723, 0.7633, 0.8042, 0.3335, 0.07199999999999995, 0.20920000000000005, 0.8002]
Final Weights : [0.0, 0.21299999999999997, 0.38970000000000005, 0.372, 0.3893, 0.43069999999999997, 0.4505, 0.5399, 0.6135999999999999, 0.6303000000000001, 0.7565999999999999, 0.8293, 0.7028, 0.5339, 0.6772, 0.7459, 0.7142999999999999, 0.47019999999999995, 0.7025, 0.6126, 0.6778, 0.7113, 0.8852, 0.8366, 1.0, 0.9723, 0.7633, 0.8042, 0.3335, 0.07199999999999995, 0.20920000000000005, 0.8002]
Saved → output